# Cách B v2: matched conditioning, with a full-budget v1 control

v1 (`jepa-guided-diffusion`) trained the adapter with c = JEPA(an independent corruption of the clean crop, the clean crop itself 20 % of the time), but denoised with c = JEPA(the same noisy photo the reverse chain starts from). The adapter treats c as a second, independent view of the image, so the photo's evidence is counted twice and the chain under-denoises: on div2k_9 it kept 2.4× more of the input noise and lost 2.6 dB.

Two adapters are trained here on the same clean crops, interleaved step by step, from the same initialisation:

| arm | conditioning during training | training timesteps |
|---|---|---|
| `v1-recipe` | v1 verbatim: `synthesize()` corruption of the clean crop, independent of x_t | 50 % t < 300, 50 % uniform |
| `matched` | c = JEPA(y), y the photo the chain would start from: σ ~ log-U(0.01, 0.35), t* = t_for_sigma(σ), t ~ U{0..t*}, x_t ~ q(x_t \| x_0), x_t* ~ q(x_t* \| x_t), y = clamp((x_t*/√ᾱ_t* + 1)/2) | the denoising range t ≤ t* |

Everything else is v1's: frozen ADM, ε-loss, 10 % null latents, AdamW 1e-4, batch 8, now 3,000 steps for each arm (v1 stopped at 1,326 on its time budget).

**Prediction.** The forward process is Markov, so x_0 ⊥ x_t* | x_t: the matched latents carry no information about x_0 beyond x_t, and the Bayes-optimal ε-predictor ignores them. The matched JEPA arm should therefore match its null arm and stop under-denoising; any gain over null can only come from approximation, not new information. The `v1-recipe` arm tests whether more training alone fixes the failure (prediction: no).

**Validation** every 250 steps on 8 held-out DIV2K images at σ ∈ {0.05, 0.1, 0.2, 0.3}: one-step Tweedie PSNR at t* and the share of the input noise kept, for ADM and each arm's null and JEPA variants, with c from the same noisy photo, as at inference.

**Evaluation:** v1's 52 cases with the same inputs and seeds (Tweedie and reverse for ADM and both arms), so the scores compare directly with v1's `scores.csv` and with the standard denoisers of `jepa-denoise-comparison`.

Inputs: `jepa_denoise_vith14_fp16.pt` (output of `jepa-denoise-finetune` v2) under `/kaggle/input` or `$INPUT_DIR`; optionally `jepa-guided-diffusion/…/scores.csv` and `jepa-denoise-comparison/…/standard_denoisers_scores.csv` there for the comparison tables. One GPU with ≥ 24 GB, or two smaller ones.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lpips"], check=True)

In [ ]:
import copy, glob, io, json, math, os, random, re, subprocess, sys, time, urllib.request, zipfile
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from torch.utils.checkpoint import checkpoint
from torchvision.io import decode_jpeg, encode_jpeg
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

# ---- knobs ----
STEPS, BS, LR = 3000, 8, 1e-4
P_DROP = 0.1                   # share of null latents (c = 0) during training, both arms
T_FOCUS, P_FOCUS = 300, 0.5    # v1 recipe only: this share of training timesteps is drawn from [0, T_FOCUS)
P_CLEAN, P_LOCAL = 0.2, 0.3    # v1 recipe only: noise synthesis for the conditioning input, as in the JEPA fine-tune
SIGMA_RANGE = (0.01, 0.35)     # matched arm: noise level of the photo the chain starts from, log-uniform
TIME_BUDGET_MIN = 240          # training stops early past this; evaluation still runs
VAL_EVERY, VAL_SIGMAS = 250, [0.05, 0.1, 0.2, 0.3]
N_EVAL, SIGMAS = 16, [0.1, 0.2, 0.3]
SAVE_IMAGES = [0, 1, 2, 3, 9]  # DIV2K cases whose outputs are saved as PNG; the real pairs are always saved
ARMS = ("v1-recipe", "matched")

assert torch.cuda.is_available(), "GPU required"
D0 = torch.device("cuda:0")                                            # ADM + adapters
D1 = torch.device("cuda:1") if torch.cuda.device_count() > 1 else D0  # I-JEPA
print([torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())], torch.__version__)
CACHE, OUT = os.environ.get("CACHE_DIR", "/tmp/guided_cache"), "outputs"
INPUT_ROOTS = ["/kaggle/input", os.environ.get("INPUT_DIR", "/workspace/inputs")]
for d in (CACHE, OUT, f"{OUT}/images"):
    os.makedirs(d, exist_ok=True)
random.seed(0); np.random.seed(0); torch.manual_seed(0)
S = 256


def find_input(pattern):
    """First file matching the glob pattern under the input roots (Kaggle-attached outputs or $INPUT_DIR), else None."""
    for root in INPUT_ROOTS:
        hits = sorted(glob.glob(f"{root}/**/{pattern}", recursive=True))
        if hits:
            return hits[0]
    return None


def download(url, path, size=None):
    """Resumable download with progress; no-op when the file is already complete."""
    have = os.path.getsize(path) if os.path.exists(path) else 0
    if size and have == size:
        return path
    if size and have > size:
        have = 0
    req = urllib.request.Request(url, headers={"Range": f"bytes={have}-"} if have else {})
    with urllib.request.urlopen(req, timeout=60) as r:
        if have and r.status != 206:
            have = 0  # server ignored the Range header: start over
        total, t0, got, mark = have + int(r.headers.get("Content-Length", 0)), time.time(), 0, 0
        with open(path, "ab" if have else "wb") as f:
            while chunk := r.read(1 << 23):
                f.write(chunk)
                got += len(chunk)
                if got - mark >= 1 << 30:
                    mark = got
                    print(f"  {(have + got) / 1e9:.1f}/{total / 1e9:.1f} GB  {got / 1e6 / (time.time() - t0):.0f} MB/s", flush=True)
    if size:
        assert os.path.getsize(path) == size, f"incomplete download: {path}"
    return path


def slug(text):
    return re.sub(r"[^a-z0-9]+", "-", text.lower()).strip("-")


def save_png(img, path):
    Image.fromarray((img.permute(1, 2, 0).numpy() * 255).round().astype(np.uint8)).save(path)

## 1 · Noise bank (training noises of the JEPA fine-tune)

In [ ]:
def geo(lo, hi, s):
    return lo * (hi / lo) ** s


def _randn(shape, x, g):
    return torch.randn(shape, generator=g, device=x.device)


def _rand(shape, x, g):
    return torch.rand(shape, generator=g, device=x.device)


def gaussian(x, s, g):
    return x + geo(0.01, 0.25, s) * _randn(x.shape, x, g)


def gaussian_mono(x, s, g):
    return x + geo(0.01, 0.25, s) * _randn((1, *x.shape[1:]), x, g)


def chroma(x, s, g):
    c = F.interpolate(_randn((1, 3, x.shape[1] // 8, x.shape[2] // 8), x, g), size=tuple(x.shape[1:]),
                      mode="bilinear", align_corners=False)[0]
    return x + geo(0.01, 0.15, s) * c / c.std()


def correlated(x, s, g):
    n = TF.gaussian_blur(_randn(x.shape, x, g), [7, 7], [1.2, 1.2])
    return x + geo(0.01, 0.2, s) * n / n.std()


def poisson(x, s, g):
    lam = geo(1000.0, 10.0, s)
    return torch.poisson(x.clamp(0, 1) * lam, generator=g) / lam


def poisson_gaussian(x, s, g):
    a, b = geo(1e-4, 1e-2, s), geo(0.005, 0.05, s)
    return x + torch.sqrt(a * x.clamp(0, 1) + b * b) * _randn(x.shape, x, g)


def salt_pepper(x, s, g):
    p, u = geo(0.002, 0.1, s), _rand(x.shape[1:], x, g)
    y = x.clone()
    y[:, u < p / 2] = 0.0
    y[:, u > 1 - p / 2] = 1.0
    return y


def dead_hot(x, s, g):
    p, u = geo(0.0005, 0.02, s), _rand(x.shape, x, g)
    y = x.clone()
    y[u < p / 2] = 0.0
    y[u > 1 - p / 2] = 1.0
    return y


def uniform(x, s, g):
    return x + geo(0.02, 0.4, s) * (2 * _rand(x.shape, x, g) - 1)


def quantize(x, s, g):
    L = round(geo(64, 4, s))
    return torch.round(x.clamp(0, 1) * (L - 1)) / (L - 1)


def jpeg(x, s, g):
    u8 = (x.clamp(0, 1) * 255).round().to(torch.uint8).cpu()
    return (decode_jpeg(encode_jpeg(u8, quality=round(geo(90, 5, s)))).float() / 255).to(x.device)


_DEMOSAIC = torch.tensor([[1.0, 2, 1], [2, 4, 2], [1, 2, 1]]) / 4


def camera(x, s, g):  # Bayer RGGB raw + shot/read noise + bilinear demosaic + gamma
    H, W = x.shape[1:]
    m = torch.zeros(3, H, W, device=x.device)
    m[0, 0::2, 0::2] = 1
    m[1, 0::2, 1::2] = 1
    m[1, 1::2, 0::2] = 1
    m[2, 1::2, 1::2] = 1
    raw = (x.clamp(0, 1) ** 2.2 * m).sum(0, keepdim=True)
    raw = raw + torch.sqrt(geo(1e-4, 1e-2, s) * raw + geo(0.002, 0.02, s) ** 2) * _randn(raw.shape, x, g)
    k = _DEMOSAIC.to(x.device).expand(3, 1, 3, 3)
    est = F.conv2d((raw * m)[None], k, padding=1, groups=3)[0] / F.conv2d(m[None], k, padding=1, groups=3)[0]
    return torch.where(m > 0, raw.expand(3, -1, -1), est).clamp(0, 1) ** (1 / 2.2)


def blur(x, s, g):
    v = geo(0.5, 3.0, s)
    k = 2 * math.ceil(3 * v) + 1
    return TF.gaussian_blur(x, [k, k], [v, v])


NOISES = {f.__name__: f for f in (gaussian, gaussian_mono, chroma, correlated, poisson, poisson_gaussian, salt_pepper,
                                  dead_hot, uniform, quantize, jpeg, camera, blur)}


def corrupt(y, kind, s, g=None):
    return NOISES[kind](y, s, g).clamp(0, 1)


def synthesize(y):
    """1-3 stacked noises, whole image or a random region; with probability P_CLEAN the image stays clean."""
    if random.random() < P_CLEAN:
        return y.clone()
    x = y
    for kind in random.sample(list(NOISES), random.choices((1, 2, 3), (0.6, 0.3, 0.1))[0]):
        x = corrupt(x, kind, random.random())
    if random.random() < P_LOCAL:
        H = y.shape[-1]
        area, ar = random.uniform(0.1, 0.5), math.exp(random.uniform(-0.7, 0.7))
        h, w = min(H, round(H * math.sqrt(area * ar))), min(H, round(H * math.sqrt(area / ar)))
        r, c = random.randint(0, H - h), random.randint(0, H - w)
        m = torch.zeros(1, H, H, device=y.device)
        m[:, r:r + h, c:c + w] = 1
        x = y * (1 - m) + x * m
    return x


def add_gaussian(x, sigma, seed=0):
    return (x + sigma * torch.randn(x.shape, generator=torch.Generator().manual_seed(seed))).clamp(0, 1)

## 2 · Data
DIV2K train (800) for the adapter, DIV2K valid (100, different source images) for evaluation; real camera-noise pairs from PolyU (noisy shot vs. the mean of many shots), cropped at native resolution.

In [ ]:
def load_zip_images(path, short):
    with zipfile.ZipFile(path) as z:
        names = sorted(n for n in z.namelist() if n.lower().endswith(".png"))

    def one(name):
        with zipfile.ZipFile(path) as z:
            im = Image.open(io.BytesIO(z.read(name))).convert("RGB")
        return torch.from_numpy(np.asarray(TF.resize(im, short)).copy()).permute(2, 0, 1).contiguous()

    with ThreadPoolExecutor(8) as ex:
        return list(ex.map(one, names))


def fetch_image(url):
    with urllib.request.urlopen(url, timeout=60) as r:
        return Image.open(io.BytesIO(r.read())).convert("RGB")


DIV2K = "https://data.vision.ee.ethz.ch/cvl/DIV2K/"
TRAIN_IMGS = load_zip_images(download(DIV2K + "DIV2K_train_HR.zip", f"{CACHE}/DIV2K_train_HR.zip", size=3_530_603_713), 320)
VALID_IMGS = load_zip_images(download(DIV2K + "DIV2K_valid_HR.zip", f"{CACHE}/DIV2K_valid_HR.zip", size=448_993_893), S)
EVAL_CLEAN = torch.stack([TF.center_crop(im, S).float() / 255 for im in VALID_IMGS[:N_EVAL]])
VAL_CLEAN = torch.stack([TF.center_crop(im, S).float() / 255 for im in VALID_IMGS[N_EVAL:N_EVAL + 8]])
CROP = T.RandomResizedCrop(S, scale=(0.4, 1.0), antialias=True)


def clean_batch(B):  # crops + horizontal flips only: ADM's prior expects upright, unshuffled photos
    ys = torch.stack([CROP(TRAIN_IMGS[random.randrange(len(TRAIN_IMGS))]) for _ in range(B)]).float() / 255
    flip = torch.rand(B) < 0.5
    ys[flip] = ys[flip].flip(-1)
    return ys.to(D0)


POLYU = "https://raw.githubusercontent.com/csjunxu/PolyU-Real-World-Noisy-Images-Dataset/master/CroppedImages/"
REAL = []
for n in ["Canon80D_8_8_3200_ball_1", "Canon5D2_5_160_6400_bicycle_10", "NikonD800_5_100_4000_flower_15", "Sony_4-5_125_3200_plant_13"]:
    noisy, clean = (TF.to_tensor(TF.center_crop(fetch_image(f"{POLYU}{n}_{k}.JPG"), S)) for k in ("real", "mean"))
    REAL.append((n, noisy, clean))
print(f"{len(TRAIN_IMGS)} train / {len(VALID_IMGS)} valid images, {len(REAL)} real noisy pairs")

## 3 · Models: frozen ADM (GPU 0) and the denoising I-JEPA target encoder (GPU 1)

In [ ]:
GD, IJ = f"{CACHE}/guided-diffusion", f"{CACHE}/ijepa"
for url, path in (("https://github.com/openai/guided-diffusion", GD), ("https://github.com/facebookresearch/ijepa", IJ)):
    if not os.path.isdir(path):
        subprocess.run(["git", "clone", "-q", "--depth", "1", url, path], check=True)
sys.path[:0] = [GD, IJ]
from guided_diffusion.script_util import create_model
from guided_diffusion.nn import timestep_embedding
import guided_diffusion.unet as gd_unet

# ADM's attention blocks always use its own checkpoint function, which differentiates w.r.t. the block weights and
# fails when they are frozen; eps_pred() checkpoints whole blocks with torch.utils.checkpoint instead.
gd_unet.checkpoint = lambda func, inputs, params, flag: func(*inputs)
from src.models import vision_transformer as vt

adm = create_model(image_size=256, num_channels=256, num_res_blocks=2, learn_sigma=True, class_cond=False,
                   attention_resolutions="32,16,8", num_head_channels=64, resblock_updown=True, use_fp16=True,
                   use_scale_shift_norm=True)  # the README's flags for 256x256_diffusion_uncond.pt
ADM_CKPT = download("https://openaipublic.blob.core.windows.net/diffusion/jul-2021/256x256_diffusion_uncond.pt",
                    f"{CACHE}/256x256_diffusion_uncond.pt", size=2_211_383_297)
adm.load_state_dict(torch.load(ADM_CKPT, map_location="cpu", weights_only=True))
adm.to(D0).eval().requires_grad_(False).convert_to_fp16()


def _sdpa_attention(self, x):  # same math as I-JEPA's attention, memory-efficient kernel
    B, n, C = x.shape
    q, k, v = self.qkv(x).reshape(B, n, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
    x = F.scaled_dot_product_attention(q, k, v, scale=self.scale)
    return self.proj(x.transpose(1, 2).reshape(B, n, C)), None


vt.Attention.forward = _sdpa_attention
found = [path for path in [find_input("jepa_denoise_vith14_fp16.pt")] if path]
assert found, "attach the output of the jepa-denoise-finetune notebook (version 2) as input data"
jepa = vt.vit_huge(patch_size=14)
jepa.load_state_dict(torch.load(found[0], map_location="cpu", weights_only=True)["target_encoder"])
jepa = jepa.half().to(D1).eval().requires_grad_(False)
JMEAN = torch.tensor([0.485, 0.456, 0.406], device=D1).view(1, 3, 1, 1)
JSTD = torch.tensor([0.229, 0.224, 0.225], device=D1).view(1, 3, 1, 1)


@torch.no_grad()
def jepa_latents(x):
    """x [B,3,S,S] in [0,1] -> [B,D,16,16]: LayerNormed target-encoder tokens of the 224 px resized image."""
    x = F.interpolate(x.to(D1).float(), size=(224, 224), mode="bilinear", antialias=True, align_corners=False)
    with torch.autocast("cuda", dtype=torch.float16):
        h = jepa((x - JMEAN) / JSTD)
    h = F.layer_norm(h.float(), (h.size(-1),))
    return h.transpose(1, 2).reshape(len(x), h.size(-1), 16, 16).to(D0)

## 4 · Adapter and conditional ADM

In [ ]:
@torch.no_grad()
def injection_points(res_list=(32, 16, 8)):
    """{ADM encoder block index: (resolution, channels)} for the first block at each resolution."""
    emb = adm.time_embed(timestep_embedding(torch.zeros(1, dtype=torch.long, device=D0), adm.model_channels))
    h, points = torch.zeros(1, 3, S, S, device=D0).type(adm.dtype), {}
    for i, m in enumerate(adm.input_blocks):
        h = m(h, emb)
        if h.shape[-1] in res_list and h.shape[-1] not in [r for r, _ in points.values()]:
            points[i] = (h.shape[-1], h.shape[1])
    return points


def _zero(m):
    nn.init.zeros_(m.weight)
    nn.init.zeros_(m.bias)
    return m


class JEPAAdapter(nn.Module):
    """JEPA latents [B,D,16,16] -> features added to ADM encoder blocks + an offset on ADM's time embedding."""

    def __init__(self, d, points, emb_dim):
        super().__init__()
        self.res = {str(i): r for i, (r, _) in points.items()}
        self.branches = nn.ModuleDict({str(i): nn.Sequential(
            nn.Conv2d(d, ch, 1), nn.GroupNorm(32, ch), nn.SiLU(), nn.Conv2d(ch, ch, 3, padding=1), nn.SiLU(),
            _zero(nn.Conv2d(ch, ch, 1))) for i, (r, ch) in points.items()})
        self.glob = nn.Sequential(nn.Linear(d, emb_dim), nn.SiLU(), _zero(nn.Linear(emb_dim, emb_dim)))

    def forward(self, c):
        feats = {}
        for i, branch in self.branches.items():
            r = self.res[i]
            feats[int(i)] = branch(c if r == c.shape[-1] else F.interpolate(c, size=(r, r), mode="bilinear", align_corners=False))
        return feats, self.glob(c.mean((2, 3)))


def eps_pred(x, t, cond=None, grad_ckpt=False):
    """ADM's UNetModel.forward (ε channels only) with adapter features added after the injection blocks."""
    emb = adm.time_embed(timestep_embedding(t, adm.model_channels))
    feats = {}
    if cond is not None:
        feats, g = cond
        emb = emb + g
    run = (lambda m, *a: checkpoint(m, *a, use_reentrant=False)) if grad_ckpt else (lambda m, *a: m(*a))
    hs, h = [], x.type(adm.dtype)
    for i, m in enumerate(adm.input_blocks):
        h = run(m, h, emb)
        if i in feats:
            h = h + feats[i].type(h.dtype)
        hs.append(h)
    h = run(adm.middle_block, h, emb)
    for m in adm.output_blocks:
        h = run(m, torch.cat([h, hs.pop()], 1), emb)
    return adm.out(h.type(x.dtype))[:, :3].float()


INJECT = injection_points()
adapter = JEPAAdapter(jepa.embed_dim, INJECT, adm.model_channels * 4).to(D0)
print("injection points {block: (px, channels)}:", INJECT, f"| adapter {sum(p.numel() for p in adapter.parameters()) / 1e6:.1f}M params")

BETAS = torch.linspace(1e-4, 0.02, 1000, dtype=torch.float64)  # ADM's linear schedule
ABAR = torch.cumprod(1 - BETAS, 0)
ABAR_D = ABAR.float().to(D0)

with torch.no_grad():  # the re-implemented forward equals ADM's, and the untrained adapter changes nothing
    xt, tt = torch.randn(2, 3, S, S, device=D0), torch.tensor([10, 500], device=D0)
    ref = adm(xt, tt)[:, :3].float()
    assert torch.allclose(eps_pred(xt, tt), ref, atol=1e-3)
    assert torch.allclose(eps_pred(xt, tt, adapter(jepa_latents(EVAL_CLEAN[:2]))), ref, atol=1e-3)

## 5 · Two arms: v1 recipe and matched conditioning
Both adapters start from the same weights (the zero-output initialisation checked above).

In [ ]:
# Muon for the adapters' weight matrices, AdamW for norms and biases; copied from laboratory/optimization/muon.py
NEWTON_SCHULZ = (3.4445, -4.7750, 2.0315)


def newton_schulz(matrix: torch.Tensor, steps: int = 5) -> torch.Tensor:
    """Approximate the orthogonal factor ``U V^T`` of ``matrix = U S V^T``; singular values end up in about [0.7, 1.2]."""
    a, b, c = NEWTON_SCHULZ
    native_bf16 = matrix.is_cuda and torch.cuda.get_device_capability(matrix.device)[0] >= 8
    x = matrix.to(torch.bfloat16 if native_bf16 else torch.float32)
    tall = x.size(-2) > x.size(-1)
    if tall:
        x = x.mT
    x = x / (x.norm(dim=(-2, -1), keepdim=True) + 1e-7)
    for _ in range(steps):
        gram = x @ x.mT
        x = a * x + (b * gram + c * gram @ gram) @ x
    return (x.mT if tall else x).to(matrix.dtype)


class Muon(torch.optim.Optimizer):
    """Muon for weight matrices (two or more dimensions), AdamW for every other parameter; decoupled weight decay."""

    def __init__(self, params, lr=3e-4, weight_decay=0.0, momentum=0.95, nesterov=True, ns_steps=5, betas=(0.9, 0.999), eps=1e-8):
        params = list(params)
        groups = [{"params": [p for p in params if p.ndim >= 2], "use_muon": True},
                  {"params": [p for p in params if p.ndim < 2], "use_muon": False}]
        defaults = dict(lr=lr, weight_decay=weight_decay, momentum=momentum, nesterov=nesterov, ns_steps=ns_steps, betas=betas, eps=eps)
        super().__init__([g for g in groups if g["params"]], defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            update = self._muon_update if group["use_muon"] else self._adamw_update
            for p in group["params"]:
                if p.grad is None:
                    continue
                direction = update(p.grad, self.state[p], group)
                p.mul_(1 - group["lr"] * group["weight_decay"])
                p.add_(direction, alpha=-group["lr"])
        return loss

    @staticmethod
    def _muon_update(grad, state, group):
        if not state:
            state["momentum_buffer"] = torch.zeros_like(grad)
        buffer = state["momentum_buffer"]
        buffer.lerp_(grad, 1 - group["momentum"])
        momentum = grad.lerp(buffer, group["momentum"]) if group["nesterov"] else buffer
        matrix = newton_schulz(momentum.reshape(len(momentum), -1), group["ns_steps"])
        return matrix.reshape_as(grad) * (0.2 * math.sqrt(max(matrix.shape)))

    @staticmethod
    def _adamw_update(grad, state, group):
        beta1, beta2 = group["betas"]
        if not state:
            state["step"] = 0
            state["exp_avg"] = torch.zeros_like(grad)
            state["exp_avg_sq"] = torch.zeros_like(grad)
        state["step"] += 1
        state["exp_avg"].lerp_(grad, 1 - beta1)
        state["exp_avg_sq"].mul_(beta2).addcmul_(grad, grad, value=1 - beta2)
        corrected = (state["exp_avg_sq"] / (1 - beta2 ** state["step"])).sqrt().add_(group["eps"])
        return state["exp_avg"] / (1 - beta1 ** state["step"]) / corrected


ADAPTERS = {arm: copy.deepcopy(adapter).train() for arm in ARMS}   # identical initialisation
OPTS = {arm: Muon(ADAPTERS[arm].parameters(), lr=LR, weight_decay=0.0) for arm in ARMS}
SCALERS = {arm: torch.amp.GradScaler("cuda") for arm in ARMS}  # ADM runs in fp16: scale the loss so its gradients do not underflow
del adapter


def t_for_sigma(sigma):  # pixel σ on [0, 1] -> step whose noise level matches (x in [-1, 1] doubles σ)
    return int((ABAR - 1 / (1 + (2 * sigma) ** 2)).abs().argmin())


def to01(z):
    return ((z + 1) / 2).clamp(0, 1).cpu()


def t_star_of(sigma):
    """t_for_sigma for a tensor of pixel sigmas (same rule, vectorised)."""
    target = 1 / (1 + (2 * sigma.double()) ** 2)
    return (ABAR[None, :] - target[:, None]).abs().argmin(1)


def v1_batch(y):
    """v1's batch: latents of an independent corruption of the clean crop, v1's timestep mix."""
    c = jepa_latents(torch.stack([synthesize(yi) for yi in y]))
    focus = torch.rand(len(y)) < P_FOCUS
    t = torch.where(focus, torch.randint(0, T_FOCUS, (len(y),)), torch.randint(0, 1000, (len(y),))).to(D0)
    eps = torch.randn_like(y)
    a = ABAR_D[t].view(-1, 1, 1, 1)
    return a.sqrt() * (2 * y - 1) + (1 - a).sqrt() * eps, t, eps, c


def matched_batch(y):
    """Latents of the photo the reverse chain would start from, on the same forward trajectory as x_t:
    x_t ~ q(x_t | x_0) and x_t* ~ q(x_t* | x_t) with t <= t*, photo = x_t* rescaled to [0, 1] and clipped like test inputs."""
    sigma = torch.exp(torch.empty(len(y)).uniform_(math.log(SIGMA_RANGE[0]), math.log(SIGMA_RANGE[1])))
    t_star = t_star_of(sigma)
    t = torch.minimum((torch.rand(len(y), dtype=torch.float64) * (t_star + 1)).long(), t_star)  # uniform on {0, ..., t*}
    t, t_star = t.to(D0), t_star.to(D0)
    a_t, a_s = ABAR_D[t].view(-1, 1, 1, 1), ABAR_D[t_star].view(-1, 1, 1, 1)
    eps = torch.randn_like(y)
    x_t = a_t.sqrt() * (2 * y - 1) + (1 - a_t).sqrt() * eps
    ratio = a_s / a_t
    x_s = ratio.sqrt() * x_t + (1 - ratio).clamp_min(0).sqrt() * torch.randn_like(y)
    photo = ((x_s / a_s.sqrt() + 1) / 2).clamp(0, 1)
    return x_t, t, eps, jepa_latents(photo)


BATCHES = {"v1-recipe": v1_batch, "matched": matched_batch}

## 6 · Validation: what inference sees
One-step Tweedie estimate at t* on 8 unseen images, with c from the same noisy photo. `kept` is the least-squares share of the input noise left in the estimate; double counting the photo shows up as JEPA keeping more than null.

In [ ]:
VAL_PHOTOS = {s: torch.stack([add_gaussian(y, s, seed=7000 + 100 * k + i) for i, y in enumerate(VAL_CLEAN)])
              for k, s in enumerate(VAL_SIGMAS)}


@torch.no_grad()
def tweedie_x0(x, t, cond):
    a = ABAR[t].item()
    eps = eps_pred(x, torch.full((len(x),), t, device=D0, dtype=torch.long), cond)
    return ((x - (1 - a) ** 0.5 * eps) / a ** 0.5).clamp(-1, 1)


@torch.no_grad()
def val_metrics(step):
    for arm in ARMS:
        ADAPTERS[arm].eval()
    clean, records = VAL_CLEAN.to(D0), []
    for s, photo in VAL_PHOTOS.items():
        photo = photo.to(D0)
        t0 = t_for_sigma(s)
        x = ABAR[t0].sqrt().float() * (2 * photo - 1)
        c = jepa_latents(photo)
        variants = {"ADM": None}
        for arm in ARMS:
            variants[f"{arm} null"] = ADAPTERS[arm](torch.zeros_like(c))
            variants[f"{arm} JEPA"] = ADAPTERS[arm](c)
        noise = photo - clean
        for name, cond in variants.items():
            err = (tweedie_x0(x, t0, cond) + 1) / 2 - clean
            records.append(dict(step=step, sigma=s, variant=name,
                                psnr=float((10 * torch.log10(1 / err.pow(2).mean((1, 2, 3)))).mean()),
                                kept=float((err.mul(noise).sum((1, 2, 3)) / noise.pow(2).sum((1, 2, 3))).mean())))
    for arm in ARMS:
        ADAPTERS[arm].train()
    return records


def val_report(records):
    v = pd.DataFrame(records).set_index(["variant", "sigma"])
    lines = []
    for s in VAL_SIGMAS:
        adm = v.loc[("ADM", s)]
        arms = " | ".join(f"{arm}: null {v.loc[(f'{arm} null', s), 'psnr'] - adm.psnr:+.3f}, JEPA {v.loc[(f'{arm} JEPA', s), 'psnr'] - adm.psnr:+.3f} dB, "
                          f"kept {v.loc[(f'{arm} JEPA', s), 'kept']:.3f}" for arm in ARMS)
        lines.append(f"    σ {s}: ADM {adm.psnr:.2f} dB, kept {adm.kept:.3f} | {arms}")
    return "\n".join(lines)

## 7 · Train both arms, interleaved on the same clean crops

In [ ]:
log, val_log, t_start = [], [], time.time()
for step in range(STEPS):
    if time.time() - t_start > TIME_BUDGET_MIN * 60:
        print(f"time budget reached at step {step}")
        break
    y = clean_batch(BS)
    row = dict(step=step)
    for arm in ARMS:
        x_t, t, eps, c = BATCHES[arm](y)
        drop = torch.rand(BS, device=D0) < P_DROP
        c = c.masked_fill(drop[:, None, None, None], 0.0)
        loss = ((eps_pred(x_t, t, ADAPTERS[arm](c), grad_ckpt=True) - eps) ** 2).mean()
        OPTS[arm].zero_grad(set_to_none=True)
        SCALERS[arm].scale(loss).backward()
        SCALERS[arm].unscale_(OPTS[arm])
        torch.nn.utils.clip_grad_norm_(ADAPTERS[arm].parameters(), 1.0)
        SCALERS[arm].step(OPTS[arm])
        SCALERS[arm].update()
        row[arm] = loss.item()
    log.append(row)
    if step % VAL_EVERY == 0 or step == STEPS - 1:
        records = val_metrics(step)
        val_log += records
        recent = pd.DataFrame(log[-VAL_EVERY:])
        mem = [round(torch.cuda.max_memory_allocated(i) / 2**30, 1) for i in range(torch.cuda.device_count())]
        print(f"step {step:5d}  train ε-MSE " + ", ".join(f"{arm} {recent[arm].mean():.4f}" for arm in ARMS)
              + f"  {(time.time() - t_start) / 60:.1f} min  peak GB {mem}\n" + val_report(records), flush=True)
STEPS_DONE, TRAIN_MIN = len(log), (time.time() - t_start) / 60
train_log, val_df = pd.DataFrame(log), pd.DataFrame(val_log)
train_log.to_csv(f"{OUT}/train_log.csv", index=False)
val_df.to_csv(f"{OUT}/validation.csv", index=False)
for arm in ARMS:
    torch.save(ADAPTERS[arm].state_dict(), f"{OUT}/adapter_{arm}.pt")

fig, axs = plt.subplots(1, 3, figsize=(18, 4.3))
for arm in ARMS:
    axs[0].plot(train_log.step, train_log[arm].rolling(100, min_periods=min(100, len(train_log))).mean(), label=arm)
axs[0].set(xlabel="step", ylabel="ε MSE (100-step mean)", title="training loss (each arm on its own timesteps)")
axs[0].legend()
for arm, style in zip(ARMS, ("--", "-")):
    for s in VAL_SIGMAS:
        p = val_df[val_df.sigma == s].pivot(index="step", columns="variant", values=["psnr", "kept"])
        axs[1].plot(p.index, p["psnr"][f"{arm} JEPA"] - p["psnr"][f"{arm} null"], style, marker="o", ms=3, label=f"{arm}, σ {s}")
        axs[2].plot(p.index, p["kept"][f"{arm} JEPA"] - p["kept"][f"{arm} null"], style, marker="o", ms=3, label=f"{arm}, σ {s}")
axs[1].set(xlabel="step", ylabel="JEPA − null, dB", title="validation: one-step Tweedie PSNR at t*")
axs[2].set(xlabel="step", ylabel="JEPA − null", title="validation: share of input noise kept")
for a in axs[1:]:
    a.axhline(0, color="black", lw=0.8)
    a.legend(fontsize=7, ncol=2)
plt.tight_layout(); plt.savefig(f"{OUT}/training.png", dpi=110); plt.show()

## 8 · Denoise v1's 52 cases: ADM and both arms, same start and same noise draws

In [ ]:
import lpips

LPIPS = lpips.LPIPS(net="alex", verbose=False).to(D0)


def scores(out, clean):
    a, b = out.permute(1, 2, 0).numpy(), clean.permute(1, 2, 0).numpy()
    return dict(psnr=peak_signal_noise_ratio(b, a, data_range=1.0),
                ssim=structural_similarity(b, a, channel_axis=-1, data_range=1.0),
                lpips=LPIPS(out[None].to(D0) * 2 - 1, clean[None].to(D0) * 2 - 1).item())


CASES = [dict(image=f"div2k_{i}", noise=f"gaussian {s}", sigma=s, clean=y, noisy=add_gaussian(y, s, seed=1000 * i + j))
         for i, y in enumerate(EVAL_CLEAN) for j, s in enumerate(SIGMAS)]
CASES += [dict(image=n, noise="real camera", sigma=float((noisy - clean).std()), clean=clean, noisy=noisy) for n, noisy, clean in REAL]
VARIANTS = ["ADM"] + [f"{arm} {kind}" for arm in ARMS for kind in ("null", "JEPA")]
SAVE_KEYS = {f"div2k_{i}" for i in SAVE_IMAGES}


@torch.no_grad()
def denoise_all(y, sigma, c, seed=0):
    """v1's denoise_variants for every variant in one batch: reverse process from t*, same start and same noise draws.
    Returns (Tweedie estimates [V,3,S,S], posterior samples [V,3,S,S]) in VARIANTS order."""
    conds = [None]
    for arm in ARMS:
        ADAPTERS[arm].eval()
        conds += [ADAPTERS[arm](torch.zeros_like(c)), ADAPTERS[arm](c)]
    feats, emb = conds[1]
    cond = ({i: torch.cat([torch.zeros_like(feats[i]) if cd is None else cd[0][i] for cd in conds]) for i in feats},
            torch.cat([torch.zeros_like(emb) if cd is None else cd[1] for cd in conds]))
    V, t0 = len(conds), t_for_sigma(sigma)
    gen = torch.Generator(device=D0).manual_seed(seed)
    x = (ABAR[t0].sqrt().float() * (2 * y.to(D0) - 1))[None].repeat(V, 1, 1, 1)
    tweedie = None
    for t in range(t0, -1, -1):
        a_t, beta_t = ABAR[t].item(), BETAS[t].item()
        a_prev = ABAR[t - 1].item() if t > 0 else 1.0
        eps = eps_pred(x, torch.full((V,), t, device=D0, dtype=torch.long), cond)
        x0 = ((x - (1 - a_t) ** 0.5 * eps) / a_t ** 0.5).clamp(-1, 1)
        if tweedie is None:
            tweedie = x0
        if t == 0:
            x = x0
            break
        mean = (a_prev ** 0.5 * beta_t / (1 - a_t)) * x0 + ((1 - beta_t) ** 0.5 * (1 - a_prev) / (1 - a_t)) * x
        noise = torch.randn((1, 3, S, S), generator=gen, device=D0).expand_as(x)  # same draw for every variant
        x = mean + (beta_t * (1 - a_prev) / (1 - a_t)) ** 0.5 * noise
    return to01(tweedie), to01(x)


rows, kept_rows, KEEP, t_start = [], [], {}, time.time()
for k, case in enumerate(CASES):
    tw, smp = denoise_all(case["noisy"], case["sigma"], jepa_latents(case["noisy"][None]), seed=k)
    outs = {"noisy input": case["noisy"]}
    for v, name in enumerate(VARIANTS):
        outs[f"{name}: Tweedie"], outs[f"{name}: reverse"] = tw[v], smp[v]
    clean, noise = case["clean"], case["noisy"] - case["clean"]
    for m, o in outs.items():
        rows.append(dict(image=case["image"], noise=case["noise"], method=m, **scores(o, clean)))
        err = o - clean
        kept = float((err * noise).sum() / (noise * noise).sum())
        kept_rows.append(dict(image=case["image"], noise=case["noise"], method=m, kept=kept,
                              mse=float(err.pow(2).mean()), uncorrelated_mse=float((err - kept * noise).pow(2).mean())))
    if case["noise"] == "real camera" or case["image"] in SAVE_KEYS:
        KEEP[(case["image"], case["noise"])] = (clean, outs)
        folder = f"{OUT}/images/{slug(case['image'])}__{slug(case['noise'])}"
        os.makedirs(folder, exist_ok=True)
        for j, (name, img) in enumerate([("clean (reference)", clean), *outs.items()]):
            save_png(img, f"{folder}/{j:02d}-{slug(name)}.png")
    if k % 4 == 0 or k == len(CASES) - 1:
        print(f"  case {k + 1}/{len(CASES)}  {(time.time() - t_start) / 60:.1f} min", flush=True)
EVAL_MIN = (time.time() - t_start) / 60
res, kept_df = pd.DataFrame(rows), pd.DataFrame(kept_rows)
res.to_csv(f"{OUT}/scores.csv", index=False)
kept_df.to_csv(f"{OUT}/retained_noise.csv", index=False)

## 9 · Results against ADM, the v1 run and the standard denoisers

In [ ]:
METHODS = list(dict.fromkeys(res.method))
for metric in ("psnr", "ssim", "lpips"):
    print(f"\nmean {metric.upper()} (" + ("lower" if metric == "lpips" else "higher") + " = better)")
    print(res.pivot_table(index="method", columns="noise", values=metric).reindex(METHODS).round(3).to_string())
print("\nshare of the input noise kept, mean over images")
print(kept_df.pivot_table(index="method", columns="noise", values="kept").reindex(METHODS).round(3).to_string())

wide = res.pivot_table(index=["noise", "image"], columns="method", values="psnr")
print("\nJEPA − null PSNR per arm (dB): mean, worst image, best image")
for arm in ARMS:
    for kind in ("reverse", "Tweedie"):
        d = wide[f"{arm} JEPA: {kind}"] - wide[f"{arm} null: {kind}"]
        for noise, g in d.groupby(level="noise"):
            print(f"  {arm:9s} {kind:7s} {noise:13s} mean {g.mean():+.3f}  worst {g.min():+.3f} ({g.idxmin()[1]})  best {g.max():+.3f}")

V1_RUN = {"adapter null: reverse": "v1 run null: reverse", "adapter JEPA: Tweedie": "v1 run JEPA: Tweedie",
          "adapter JEPA: reverse": "v1 run JEPA: reverse"}
ADM_GAP, comparison = {}, [res]
v1_path, standard_path = find_input("jepa-guided-diffusion/**/scores.csv"), find_input("jepa-denoise-comparison/**/standard_denoisers_scores.csv")
if v1_path:
    v1 = pd.read_csv(v1_path).replace({"method": {"ADM Tweedie": "ADM: Tweedie", "ADM reverse": "ADM: reverse"}})
    merged = res.merge(v1, on=["image", "noise", "method"], suffixes=("", "_v1"))
    ADM_GAP = {m: float((g.psnr - g.psnr_v1).abs().max()) for m, g in merged.groupby("method")}
    print("\nre-run vs the v1 run on identical inputs and seeds, max |Δ PSNR| (dB):", {m: round(v, 4) for m, v in ADM_GAP.items()})
    comparison.append(v1[v1.method.isin(list(V1_RUN))].replace({"method": V1_RUN}))
if standard_path:
    comparison.append(pd.read_csv(standard_path))
ALL = pd.concat(comparison, ignore_index=True)
ALL.to_csv(f"{OUT}/all_scores.csv", index=False)
SHOW = ["noisy input", "ADM: Tweedie", "ADM: reverse", *V1_RUN.values(),
        *[f"{v}: {kind}" for v in VARIANTS[1:] for kind in ("Tweedie", "reverse")], "BM3D", "DRUNet", "Restormer", "SCUNet"]
SHOW = [m for m in SHOW if m in set(ALL.method)]
for metric in ("psnr", "lpips"):
    print(f"\nall methods, mean {metric.upper()}")
    print(ALL.pivot_table(index="method", columns="noise", values=metric).reindex(SHOW).round(3).to_string())

## 10 · Figures: per-image effect, side by side, and generation from pure noise

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 4.3))
noises = list(dict.fromkeys(res.noise))
for a, metric, label in zip(axs, ("psnr", "lpips"), ("JEPA − null PSNR, dB (reverse)", "JEPA − null LPIPS (reverse)")):
    w = res.pivot_table(index=["noise", "image"], columns="method", values=metric)
    for shift, arm, color in ((-0.18, ARMS[0], "tab:orange"), (0.18, ARMS[1], "tab:blue")):
        d = w[f"{arm} JEPA: reverse"] - w[f"{arm} null: reverse"]
        for p, noise in enumerate(noises):
            vals = d.loc[noise].to_numpy()
            a.scatter(p + shift + np.linspace(-0.1, 0.1, len(vals)), vals, s=16, color=color, label=arm if p == 0 else None)
    a.axhline(0, color="black", lw=0.8)
    a.set_xticks(range(len(noises))); a.set_xticklabels(noises); a.set_ylabel(label); a.legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT}/per_image.png", dpi=110); plt.show()

PANELS = ["noisy input", "ADM: reverse", f"{ARMS[0]} JEPA: reverse", f"{ARMS[1]} null: reverse", f"{ARMS[1]} JEPA: reverse", f"{ARMS[1]} JEPA: Tweedie"]
picked = [key for key in KEEP if key[1] == "gaussian 0.2"] + [key for key in KEEP if key[1] == "real camera"][:1]
if picked:
    shown = res.set_index(["image", "noise", "method"])
    fig, axs = plt.subplots(len(picked), len(PANELS) + 1, figsize=(2.3 * (len(PANELS) + 1), 2.45 * len(picked)), squeeze=False)
    for r, key in enumerate(picked):
        clean, outs = KEEP[key]
        for j, (name, img) in enumerate([("clean", clean)] + [(m, outs[m]) for m in PANELS]):
            title = name if j == 0 else f"{name}\n{shown.loc[(*key, name)].psnr:.2f} dB | LPIPS {shown.loc[(*key, name)].lpips:.3f}"
            axs[r, j].imshow(img.permute(1, 2, 0).numpy()); axs[r, j].set_title(title, fontsize=7)
            axs[r, j].set_xticks([]); axs[r, j].set_yticks([])
        axs[r, 0].set_ylabel(f"{key[0][:14]}\n{key[1]}", fontsize=8)
    plt.tight_layout(); plt.savefig(f"{OUT}/overview.png", dpi=100); plt.show()

GEN_STEPS = 50
@torch.no_grad()
def generate(adapter, c, seed, steps=GEN_STEPS):
    adapter.eval()
    fj, gj = adapter(c)
    fn, gn = adapter(torch.zeros_like(c))
    cond = ({i: torch.cat([fn[i], fj[i]]) for i in fj}, torch.cat([gn, gj]))
    x = torch.randn((1, 3, S, S), generator=torch.Generator(device=D0).manual_seed(seed), device=D0).repeat(2, 1, 1, 1)
    ts = torch.linspace(999, 0, steps).round().long().tolist()
    for k, t in enumerate(ts):
        a_t = ABAR[t].item()
        a_prev = ABAR[ts[k + 1]].item() if k + 1 < len(ts) else 1.0
        eps = eps_pred(x, torch.full((2,), t, device=D0, dtype=torch.long), cond)
        x0 = ((x - (1 - a_t) ** 0.5 * eps) / a_t ** 0.5).clamp(-1, 1)
        x = a_prev ** 0.5 * x0 + (1 - a_prev) ** 0.5 * eps
    return to01(x0)  # [null, JEPA]


NGEN = min(4, len(EVAL_CLEAN))
fig, axs = plt.subplots(NGEN, 2 + 2 * len(ARMS), figsize=(2.6 * (2 + 2 * len(ARMS)), 2.65 * NGEN), squeeze=False)
for r in range(NGEN):
    clean = EVAL_CLEAN[r]
    noisy = add_gaussian(clean, 0.2, seed=r)
    c = jepa_latents(noisy[None])
    panels = [("clean", clean), ("noisy σ=0.2 (JEPA input)", noisy)]
    for arm in ARMS:
        g0 = generate(ADAPTERS[arm], c, seed=0)
        panels += [(f"{arm}: null, seed 0", g0[0]), (f"{arm}: JEPA, seed 0", g0[1])]
    for a, (title, img) in zip(axs[r], panels):
        a.imshow(img.permute(1, 2, 0)); a.set_title(title, fontsize=8); a.axis("off")
plt.tight_layout(); plt.savefig(f"{OUT}/generation.png", dpi=100); plt.show()

In [ ]:
with open(f"{OUT}/results.json", "w") as f:
    json.dump(dict(config=dict(STEPS=STEPS, BS=BS, LR=LR, OPTIMIZER="Muon", P_DROP=P_DROP, T_FOCUS=T_FOCUS, P_FOCUS=P_FOCUS, P_CLEAN=P_CLEAN,
                               SIGMA_RANGE=SIGMA_RANGE, ARMS=ARMS, steps_done=STEPS_DONE, inject=INJECT,
                               gpus=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())], torch=torch.__version__),
                   minutes=dict(training=TRAIN_MIN, evaluation=EVAL_MIN), rerun_gap_db=ADM_GAP,
                   summary=res.groupby(["noise", "method"], sort=False)[["psnr", "ssim", "lpips"]].mean().reset_index().to_dict("records"),
                   kept=kept_df.groupby(["noise", "method"], sort=False)["kept"].mean().reset_index().to_dict("records"),
                   validation=val_df.to_dict("records")), f, indent=1, default=float)
open(f"{OUT}/DONE", "w").write(time.strftime("%Y-%m-%d %H:%M:%S"))
print(sorted(os.listdir(OUT)))